# TSMC SEC 6-K 原始提交接收时钟预检

冻结三个原始完整提交文件，核对 EDGAR 头部接受时间和发行人 CIK，原件备份 NAS。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,re
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_tsmc_sec_6k_guidance_source_probe_20261023';raw=root/'data/snapshots/stage1_tsmc_sec_6k_guidance_source_probe_20261023';plan=json.loads((card/'acceptance_clock_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'result.json')==plan['parent_body_result_sha256'] and not (card/'acceptance_clock_state.json').exists()
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'acceptance_clock_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests'],1):
 body=raw/f'submission_{i}.txt';headers=raw/f'submission_{i}.headers';state['GET_attempts']+=1;(card/'acceptance_clock_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'anchor':spec['anchor'],'accession':spec['accession'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'anchor':spec['anchor'],'accession':spec['accession'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:300]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None)
 receipts.append(receipt);(card/'acceptance_capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'anchor_{i}_transport';break
 try:
  b=body.read_bytes();head=b[:12000].decode('utf-8',errors='replace');assert '<SEC-DOCUMENT>' in head.upper();accepted=re.search(r'ACCEPTANCE-DATETIME>\s*(\d{14})',head);filed=re.search(r'FILED AS OF DATE:\s*(\d{8})',head);cik=re.search(r'CENTRAL INDEX KEY:\s*0*(\d+)',head);assert accepted and filed and cik and cik.group(1)=='1046179',(bool(accepted),bool(filed),bool(cik));assert spec['accession'].replace('-','') in head or spec['accession'] in head
  checks.append({'anchor':spec['anchor'],'accession':spec['accession'],'raw_sha256':sha(body),'accepted_timestamp_EDGAR_header':accepted.group(1),'filed_date_EDGAR_header':filed.group(1),'CIK':'0001046179','bytes':len(b)});(card/'acceptance_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=f'anchor_{i}_identity_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.name.startswith('submission_')];manifest={'plan_sha256':sha(card/'acceptance_clock_plan.json'),'files':files,'receipts_sha256':sha(card/'acceptance_capture_receipts.json')};(card/'acceptance_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='tsmc-sec6k-clock-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('acceptance_clock_plan.json','acceptance_capture_receipts.json','acceptance_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='tsmc-sec6k-clock-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'three_SEC_original_submissions_acceptance_header_passed_full_history_pending' if not failure and len(checks)==3 else 'bounded_SEC_acceptance_clock_partial_or_failed','failure':failure,'plan_sha256':sha(card/'acceptance_clock_plan.json'),'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'full_history_first_public_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'acceptance_clock_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'acceptance_clock_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'accepted':[x['accepted_timestamp_EDGAR_header'] for x in checks],'NAS':digest})


{'status': 'three_SEC_original_submissions_acceptance_header_passed_full_history_pending', 'failure': None, 'GETs': 3, 'accepted': ['20110728063534', '20220414060233', '20260716074543'], 'NAS': 'aef18786ad2ac993c7461136e1ef79b2e902fc4cd0034ff546f43f7d91a00a89'}


## 中文结论

三期完整提交的 CIK、accession、原始接收头通过；使用日期级下一信号日，不把 API 带 Z 时间当真实 UTC。 详见 `conclusion.md`。